# Credit Risk ML Pipeline (Colab + GCP Compatible)

Full PySpark ML pipeline including environment setup, data loading from GCS,
EDA, cleaning, feature engineering, training, and evaluation.


In [ ]:
if 'google.colab' in str(get_ipython()):
    print('Running in Colab')
    RunningInColab = True
else:
    print('Not running in Colab')
    RunningInColab = False


In [ ]:
try:
    from pyspark.sql import SparkSession
    print('PySpark already available')
except:
    print('Installing PySpark...')
    !pip install pyspark


In [ ]:
if RunningInColab:
    from google.colab import auth
    auth.authenticate_user()
    print('Authenticated with GCP')


In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('CreditRiskProject').getOrCreate()
print('Spark Version:', spark.version)


In [ ]:
bucket = 'jkim569_data_for_gcp_labs'
path = 'data_for_project_lab/small_dataset.csv'
full_path = f'gs://{bucket}/{path}'

print('Reading:', full_path)
df = spark.read.option('header','true').option('inferSchema','true').csv(full_path)
df.printSchema()
df.show(5)


In [ ]:
from pyspark.sql.functions import col, isnan, when, count

print('Total rows:', df.count())
df.groupBy('Loan_Default_Status').count().show()

missing_counts = df.select([
    count(when(col(c).isNull() | isnan(c) | (col(c) == ''), c)).alias(c)
    for c in df.columns
])
missing_counts.show(truncate=False)


In [ ]:
from pyspark.ml.feature import Imputer

num_cols = ['Credit_Score','Annual_Income','Existing_Debt','Loan_Amount',
            'Debt_to_Income_Ratio','Transaction_Behavior_Score']

for c in num_cols:
    df = df.withColumn(c, col(c).cast('double'))

imputer = Imputer(inputCols=num_cols,
                  outputCols=[c + '_imp' for c in num_cols])

df = imputer.fit(df).transform(df)

df = df.fillna({'Employment_Status':'Unknown',
                'Gender':'Unknown',
                'Loan_Purpose':'Unknown'})

for orig in num_cols:
    df = df.drop(orig).withColumnRenamed(orig + '_imp', orig)

df.printSchema()


In [ ]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline

cat_cols = ['Gender','Employment_Status','Loan_Purpose']

indexers = [StringIndexer(inputCol=c,
                          outputCol=c + '_idx',
                          handleInvalid='keep') for c in cat_cols]

pipeline = Pipeline(stages=indexers)
df = pipeline.fit(df).transform(df)

feature_cols = ['Customer_Age','Annual_Income','Credit_Score','Existing_Debt',
                'Loan_Amount','Loan_Term_Months','Debt_to_Income_Ratio',
                'Transaction_Behavior_Score'] + [c + '_idx' for c in cat_cols]

assembler = VectorAssembler(inputCols=feature_cols,
                            outputCol='features',
                            handleInvalid='keep')

df = assembler.transform(df)
df = df.withColumn('label', col('Loan_Default_Status').cast('double'))

df.select('features','label').show(3, truncate=False)


In [ ]:
train_df, val_df = df.randomSplit([0.8, 0.2], seed=42)
print('Train:', train_df.count(), 'Validation:', val_df.count())


In [ ]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(featuresCol='features', labelCol='label', maxIter=20)
model_lr = lr.fit(train_df)
pred_lr = model_lr.transform(val_df)
pred_lr.select('label','probability','prediction').show(5)


In [ ]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(featuresCol='features', labelCol='label', numTrees=100)
model_rf = rf.fit(train_df)
pred_rf = model_rf.transform(val_df)
pred_rf.select('label','probability','prediction').show(5)


In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.mllib.evaluation import MulticlassMetrics

def compute_metrics(pred_df):
    preds_and_labels = pred_df.select('prediction','label').rdd.map(lambda r: (float(r['prediction']), float(r['label'])))
    metrics = MulticlassMetrics(preds_and_labels)
    acc = metrics.accuracy
    precision = metrics.precision(1.0)
    recall = metrics.recall(1.0)
    f1 = metrics.fMeasure(1.0)
    evaluator = BinaryClassificationEvaluator(rawPredictionCol='rawPrediction',
                                               labelCol='label',
                                               metricName='areaUnderROC')
    roc_auc = evaluator.evaluate(pred_df)
    return {'accuracy': acc, 'precision': precision, 'recall': recall, 'f1': f1, 'roc_auc': roc_auc}

print('Logistic Regression Metrics:')
print(compute_metrics(pred_lr))

print('Random Forest Metrics:')
print(compute_metrics(pred_rf))
